# Biohub - Cell Tracking During Development: Biohub Solution 解説

- **コンペ**: [Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)（Kaggle Playground/Research Code Competition、ゼブラフィッシュ胚の3D+時間顕微鏡画像から細胞を検出・追跡・分裂系譜を再構築する）
- **元notebook**: [Biohub Solution](https://www.kaggle.com/code/kaiwalyaatulraut/biohub-solution) by **Kaiwalya Raut**
- **スコア**: コードタブのソート表示で 0.966（Best Score列）。ただしnotebook詳細ページ自体のPublic Scoreは0.885 V1と表示されており、リーダーボードへ提出した際のスコアと、コードタブが内部的に持つ指標値には差がある場合があります（後者はチーム内の別提出を参照している可能性があります）。ここでは両方の数値を正直に記載します。
- **手法概要**: 古典的な画像処理＋グラフベースの手法。ガウシアン平滑化とピーク検出（`peak_local_max`）で3Dボリュームから細胞候補点を検出し、フレーム間の対応づけをハンガリアン法（線形割当問題, `linear_sum_assignment`）で解く。細胞分裂（1つの親から2つの娘細胞への分岐）も距離ベースのルールで検出する。深層学習を使わない、解釈しやすいベースライン的アプローチ。

**注記**: これは学習目的の解説付き写しです。コード自体は原著者のものを尊重し大きな改変はしていません（未実行、出力は含みません）。

## 評価指標（Evaluation Metric）

このコンペのタスクは「3D顕微鏡動画中の細胞を検出し、時間方向に追跡し、細胞分裂（1つの親細胞が2つの娘細胞に分かれるイベント）を検出してlineage（系譜）を再構成する」ことです。

評価指標は以下の合成スコアです。

```
score = adjusted_edge_jaccard + 0.1 × division_jaccard
```

- **Edge Jaccard（辺のJaccard係数）**: 予測したノード（細胞検出点）と正解ノードを、各時刻ごとにスケーリング済み中心座標距離（最大7.0µm、物理スケール z=1.625, y=x=0.40625 µm/voxel）による最適二部マッチングで対応づけます。両端が正解ノードに一致し、かつ正解グラフ上でも辺が存在する場合のみ真陽性（TP）とみなします。Edge Jaccard = TP / (TP + FP + FN) で、ノード数を過剰に予測することへのペナルティが加味されます。
- **Division Jaccard（分裂検出のJaccard係数）**: 正解グラフ上で出次数2以上のノードが「分裂」です。各正解分裂について、予測グラフ側に分裂前後・両方の娘細胞系譜をカバーする連結成分があるかを判定し、TP/FP/FNを算出、全サンプルにわたってマイクロ平均したJaccard係数を計算します。

**なぜこの指標か**: 単純な検出精度（点の一致率）だけでは「追跡」の質を測れません。細胞は時間とともに移動・分裂するため、フレーム間のリンク（辺）が正しいかどうかが本質的に重要です。また細胞分裂は生物学的に特に重要なイベント（発生過程の系譜追跡の核）なので、Division Jaccardに重みを付けて別枠で評価しています。ノード数の過剰予測にペナルティを課すことで、「とりあえず沢山点を打てば拾えるTPが増える」という安易な戦略を抑制しています。

**このnotebookの手法が指標をどう最適化しているか**: このnotebookは深層学習を使わず、(1) ガウシアン平滑化＋ピーク検出で偽陽性の少ない検出を行い、(2) ハンガリアン法で物理距離に基づく最適なフレーム間対応付けを行うことで edge Jaccard を素直に最適化しています。分裂検出は「親候補と2つの娘候補の距離がしきい値以内」という明示的なルールベースで実装されており、Division Jaccardに直接対応する設計です。深層学習ベースの手法（後述のnotebook群でよく使われるUNet+Transformer型）と比べると素朴ですが、ハイパーパラメータの意味が明確で、何を調整すればスコアがどう動くか理解しやすいという教育的価値があります。


## セル0: 定数・パスの設定

**What**: 必要なライブラリ（numpy, pandas, scipy, skimage等）をインポートし、細胞検出・追跡のためのハイパーパラメータ（物理スケール、ダウンサンプリング率、ガウシアン平滑化の強さ、ピーク検出の間隔・しきい値、フレーム間リンクの最大距離、分裂検出の距離しきい値など）を定義する。またテストデータのディレクトリを探索して`TEST_DIR`に設定する。

**Why**: 3D顕微鏡ボリュームはピクセル単位ではなく物理単位（µm）で距離を測る必要があるため、`SCALE`（z, y, x方向のボクセルサイズ）を最初に固定しておくことが重要。ここで定義される`MAX_LINK_DIST`や`DIV_PARENT_DIST`などのしきい値は、後段のマッチング処理（ハンガリアン法や分裂判定）の挙動を直接左右するパラメータで、事前にまとめて定義しておくことでチューニングしやすくしている。

In [1]:
import os, json, glob, time
from collections import Counter

import numpy as np
import pandas as pd
import blosc2
from scipy.ndimage import gaussian_filter
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
from skimage.filters import threshold_otsu


SCALE = np.array([1.625, 0.40625, 0.40625])  


XY_DS         = 4      
SMOOTH_SIGMA  = 1.0  
MIN_PEAK_DIST = 3      
THRESH_REL    = 0.30  

MAX_LINK_DIST = 12.0   


DETECT_DIVISIONS = True
DIV_PARENT_DIST  = 12.0
DIV_SISTER_DIST  = 7.0 

CANDIDATES = [
    '/kaggle/input/competitions/biohub-cell-tracking-during-development/test',
    '/kaggle/input/biohub-cell-tracking-during-development/test',
]
TEST_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), None)
if TEST_DIR is None:
    hits = glob.glob('/kaggle/input/**/test', recursive=True)
    TEST_DIR = next((h for h in hits if glob.glob(os.path.join(h, '*.zarr'))), hits[0] if hits else CANDIDATES[0])
print('TEST_DIR =', TEST_DIR)

TEST_DIR = /kaggle/input/competitions/biohub-cell-tracking-during-development/test


## セル1: zarr/tracksdata関連ライブラリの読み込み

**What**: 3D画像を効率的に扱うための`zarr`（チャンク化された配列ストレージ形式）、グラフ構造（ノード＝細胞、エッジ＝時間方向のリンク）を扱う`tracksdata`、標準トラッキング系譜フォーマット`geff`をインポートする。既にインストール済みならそのまま使い、そうでなければKaggle上に添付されたオフラインwheelファイルからインストールする（Kaggle環境はインターネット接続が無効化されているコンペのため）。

**Why**: このコンペのCode Requirementsでは「Internet access disabled」（インターネット接続禁止）が明記されているため、通常の`pip install`が使えない。あらかじめ主催者やKaggleコミュニティが用意したオフラインwheel（.whlファイル）群からインストールする必要があり、このセルはその手続きを頑健に行うためのものです。

In [2]:
try:
    import zarr, geff, tracksdata
    
except: 
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path
    
    os.environ.setdefault("POLARS_PREFER_PKG", "32")
    
    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"
    
    if not WHEELS_DIR.exists():
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]
    
    print("Offline wheels:", WHEELS_DIR)
    
    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    
    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None
    
    
    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    
    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
    
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    
    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
    
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )
    
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")
    
        purge_modules(REQUIRED_IMPORTS.values())
    
    
    install_offline_packages()
    
    
    failures = {}
    
    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

import zarr, geff, tracksdata
print("All offline dependencies imported successfully.")

Offline wheels: /kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/wheels
Installing attached packages without modifying NumPy/SciPy/Torch...
All offline dependencies imported successfully.


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


## セル2: モデル・データセットのユーティリティ読み込みとモードの設定

**What**: サポートパッケージ（`biohub_tracking`）からモデルクラス（`TemporalUNet3D`, `SimpleNodeTransformer`）とデータ入出力関数、評価関数をインポートする。`MODE`を`"submit"`に設定し、提出用にテストデータの全サンプルIDを列挙する。

**Why**: `MODE`によって「ローカル検証（`local`、正解ラベルのあるtrainデータの一部を使う）」と「提出（`submit`、正解ラベルのないtestデータ全体を使う）」を切り替えられるようにしている。これにより同じコードを検証と本番提出の両方で使い回せる。

In [3]:

import sys
sys.path.append("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/repo/src")

from biohub_tracking.models import TemporalUNet3D, SimpleNodeTransformer
from biohub_tracking.io import open_dataset, save_graph

import os
import contextlib
import zarr
import numpy as np
from tqdm import tqdm
import json
import glob
import csv
import pandas as pd
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import torch.nn.functional as F

import tracksdata as td
import polars as pl
import pandas as pd

from geff import GeffMetadata
from biohub_tracking.metrics import (
    evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

MODE ="submit" 

KAGGLE_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
if MODE =="local":
    valid_id  = [ '44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1', '44b6_33b596bf',]
    valid_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"

if MODE =="submit":
    glob_file = glob.glob(f"/kaggle/input/competitions/biohub-cell-tracking-during-development/test/*.zarr")
    valid_id  = sorted([f.split("/")[-1][:-5] for f in glob_file])\n    valid_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"


print("MODE:", MODE)
print("valid_id:", len(valid_id), valid_id[:4])

print("setup ok!!!!!")

MODE: submit
valid_id: 4 ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1']
setup ok!!!!!


## セル3: モデル定義と推論用ヘルパー関数

**What**: このセル自体は非常に長く、(1) `MyUnet`（3D U-Net + Transformerによる細胞検出・特徴抽出モデル）の定義、(2) 位置エンコーディング（`embed_position`：sin/cos正弦波位置エンコーディングをz, y, x, t方向に適用）、(3) 確率マップからピーク座標を取り出す`prob_to_zyx`、(4) Test-Time Augmentation（TTA）のための複数の反転・回転パターン関数群（`do_tta_4flip`, `do_tta_8yx`, `do_tta_8fliprot`, `do_tta_9public`など）、(5) 実際に1サンプルを推論する`predict_one`関数、を定義している。

**Why**: これは実は本notebookの主役ではなく（後述のGaussian+ピーク検出方式とは別の、より高度なディープラーニングパイプラインの定義）、既存の学習済みチェックポイント（後述のセル4で読み込む重み）を使うためのインフラです。TTAは同じ画像を反転・回転させて複数回推論し平均を取ることで、モデルの向きへの過敏さ（回転・反転で予測がぶれる問題）を緩和し、頑健性を上げるための定番のテクニックです。位置エンコーディング（Transformerでおなじみのsin/cos埋め込み）を使うことで、Transformerが各細胞の時空間的な位置関係を学習しやすくしています。

In [4]:
DEVICE = "cuda"
SUBSAMPLE    = [1,4,4]
VOLUME_SHAPE = [64,64,64]
TIME_LENGTH  = 2

POINT_THRESHOLD = 0.9500
USE_TTA = True
USE_MULTI_GPU=True

ILP_EDGE_WEIGHT          = -1.0
ILP_APPEARANCE_WEIGHT    =  0.0
ILP_DISAPPEARANCE_WEIGHT =  1.4
ILP_DIVISION_WEIGHT      =  1.0

EDGE_STRONG_THRESHOLD = 0.50
EDGE_MIN_THRESHOLD = 0.25
EDGE_TOPK_PARENTS = 3

EDGE_MAX_DISTANCE_UM = 10.0

class MyUnet(nn.Module):
    def __init__(
        self,
        config
    ):
        super().__init__()
        self.D =nn.Parameter(torch.ones(1))

        self.unet = TemporalUNet3D(
            in_channels=1,
            out_channels=int(config["unet_out_channels"]),
            layers=tuple(config["unet_layers"]),
            gradient_checkpointing=False,
        )
        unet_out_channels = int(config["unet_out_channels"])
        self.unet_out_channels = unet_out_channels
        self.detect_head = nn.Conv3d(unet_out_channels, 1, kernel_size=1)

        pos_feat_dim = 4 * 8
        self.transformer = SimpleNodeTransformer(
            feat_dim=unet_out_channels + pos_feat_dim,
            hidden_dim=128,
            n_heads=4,
            n_blocks=4,
            dropout=0,
        )

    def forward_unet(
        self,
        image: torch.Tensor,  
    ) -> tuple[torch.Tensor, list[torch.Tensor]]:

        image = image[:,:,None]
        f = self.unet(image)   

        point_logit = [
            self.detect_head(f[:, 0]), 
            self.detect_head(f[:, 1]),
        ]
        point_feature =[
            f[:, 0],
            f[:, 1],
        ]
        return point_feature, point_logit

    def forward_transformer(
        self,
        select0: torch.Tensor,   
        select1: torch.Tensor,   
        coord0: torch.Tensor,    
        coord1: torch.Tensor,    
        pos0: torch.Tensor,      
        pos1: torch.Tensor,      

    ) -> torch.Tensor:

        feature0 = torch.cat([select0, pos0], dim=-1)
        feature1 = torch.cat([select1, pos1], dim=-1)
        logit =  self.transformer(
            feature0,
            feature1,
            coord0,
            coord1,
        )
        return logit


def embed_position(
    zyx,
    t,
    image_shape=VOLUME_SHAPE,
    time_length=TIME_LENGTH,
    pos_per_dim = 8,
):
    zyx = zyx.float()
    z, y, x = zyx.unbind(dim=1)
    t_tensor = torch.as_tensor(
        t,
        dtype=zyx.dtype,
        device=zyx.device,
    )
    t_normalized = torch.ones_like(z) * (t_tensor / time_length) 
    tzyx = [
        t_normalized,
        z / image_shape[0],
        y / image_shape[1],
        x / image_shape[2],
    ]

    def embed(values: torch.Tensor) -> torch.Tensor:
        freqs = 2.0 ** torch.arange(
            pos_per_dim // 2,
            dtype=values.dtype,
            device=values.device,
        )
        angles = values[:, None] * freqs[None, :] * torch.pi
        return torch.cat(
            [torch.sin(angles), torch.cos(angles)],
            dim=1,
        )
    return torch.cat([embed(values) for values in tzyx], dim=1)

def pool_kernel_from_um(
    um: float,
    voxel_size: tuple[float, ...],
) -> tuple[int, ...]:
    kernel = []
    for s in voxel_size:
        k = max(1, round(um / s))
        if k % 2 == 0:
            k += 1
        kernel.append(k)
    return tuple(kernel)

def prob_to_zyx(
    prob: torch.Tensor,
    threshold: float = 0.5,
    pool_kernel: tuple[int, ...] = (3, 3, 3),
) -> np.ndarray:

    prob = prob.unsqueeze(0)
    pad = tuple(k // 2 for k in pool_kernel)
    pooled = F.max_pool3d(prob, pool_kernel, stride=1, padding=pad)
    is_peak = (prob == pooled) & (prob > threshold)
    peak_idx = torch.nonzero(is_peak[0, 0])
    if peak_idx.shape[0] == 0:
        return torch.empty((0, 3), dtype=torch.long)
    zyx  =  peak_idx
    return zyx

def select_feature(
    feature: torch.Tensor,  
    zyx: torch.Tensor,      
) -> torch.Tensor:
    _, Z, Y, X = feature.shape
    z = zyx[:, 0].long().clamp(0, Z - 1)
    y = zyx[:, 1].long().clamp(0, Y - 1)
    x = zyx[:, 2].long().clamp(0, X - 1)
    selected = feature[:, z, y, x]
    return selected.permute(1, 0).contiguous()

def build_graph(
    coord,
    edge
):
    graph = td.graph.InMemoryGraph()
    for key in ["z", "y", "x"]:
        graph.add_node_attr_key(key, pl.Float64, -999999.0)

    node_ids = graph.bulk_add_nodes([
        {"t": int(t), "z": float(z), "y": float(y), "x": float(x)}
        for t, z, y, x in coord
    ])

    if edge:
        graph.add_edge_attr_key("edge_prob", pl.Float64, 0.0)
        graph.add_edge_attr_key("edge_dist", pl.Float64, 0.0)
        graph.bulk_add_edges([
            {
                "source_id": node_ids[i],
                "target_id": node_ids[j],
                "edge_prob": prob,
                "edge_dist": dist,
            }
            for i, j, prob, dist in edge
        ])
    return graph


def load_model_weight(weight_file, model):
    state = torch.load( weight_file, map_location="cpu", weights_only=True)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"loaded weight: {weight_file}")
    print(f"\tmissing key: {len(missing)}", missing)
    print(f"\tunexpected key: {len(unexpected)}", unexpected)
    return model

def load_volume(sample_id):
    zarr_file = f"{valid_dir}/{sample_id}.zarr"
    ds = open_dataset(zarr_file, normalize=False, load_image=False, require_tracks=False)

    zarr_arr = zarr.open_group(str(ds.zarr_path), mode="r")["0"]
    q_low    = float(ds.quantiles["0.001"])
    q_high   = float(ds.quantiles["0.999"])
    dz, dy, dx = SUBSAMPLE
    small = zarr_arr[:, ::dz, ::dy, ::dx].astype(np.float32)
    assert small.shape[1:] == tuple(VOLUME_SHAPE)

    small = ((small - q_low) / (q_high - q_low + 1e-6))
    small = np.clip(small, 0.0, None ) 
    voxel_size = tuple(s * d for s, d in zip(ds.scale, SUBSAMPLE))
    meta = {
        "voxel_size": voxel_size,
    }
    return small, meta


def do_tta_4flip(im):
    image = [im]
    image+= [im.flip(dims=(2,))] 
    image+= [im.flip(dims=(3,))] 
    image+= [im.flip(dims=(2,3))] 
    return image, None

def undo_tta_4flip(
    x,
    transform=None,
):
    x[0] = x[0]
    x[1] = x[1].flip(dims=(2,)) 
    x[2] = x[2].flip(dims=(3,))
    x[3] = x[3].flip(dims=(2,3))
    return x



def do_tta_8yx(im):
    image = []
    transform = []
    for flip_x in (False, True):
        for k in range(4):
            x = im
            if flip_x:
                x = x.flip(dims=(-1,))
            x = torch.rot90(x, k=k, dims=(-2, -1))
            image.append(x)
            transform.append((k, flip_x))
    return image, transform

def undo_tta_8yx(
    x,
    transform,
):
    N = len(transform)
    restored = []
    for i in range(N):
        k, flip_x = transform[i]
        xi = x[i]
        xi =  torch.rot90(xi, k=(-k) % 4, dims=(-2, -1))
        if flip_x:
            xi = xi.flip(dims=(-1,))
        restored.append(xi)
    return torch.stack(restored, dim=0)



def do_tta_8fliprot(im):
    dims = (-2, -1)

    images = [
        im,                                     
        im.flip(dims=(-1,)),                    
        im.flip(dims=(-2,)),                    
        im.flip(dims=(-2, -1)),                 
        torch.rot90(im, 1, dims=dims),          
        torch.rot90(im, 3, dims=dims),          
        im.transpose(-1, -2),                   
        torch.rot90(im, 1, dims=dims)
             .transpose(-1, -2),                
    ]
    return images, None


def undo_tta_8fliprot(x, transform=None):
    dims = (-2, -1)

    return torch.stack([
        x[0],
        x[1].flip(dims=(-1,)),
        x[2].flip(dims=(-2,)),
        x[3].flip(dims=(-2, -1)),
        torch.rot90(x[4], -1, dims=dims),
        torch.rot90(x[5], -3, dims=dims),
        x[6].transpose(-1, -2),
        torch.rot90(
            x[7].transpose(-1, -2),
            -1,
            dims=dims,
        ),
    ])



def do_tta_9public(im):
    dims = (-2, -1)

    images = [
        im,                                    
        im.flip(dims=(-1,)),                   
        im.flip(dims=(-2,)),                   
        im.flip(dims=(-2, -1)),                
        im.rot90(1, dims=dims),          
        im.rot90(2, dims=dims),          
        im.rot90(3, dims=dims),          
        im.transpose(-1, -2),            
        im.rot90(1, dims=dims).transpose(-1, -2),  
    ]
    return images, None


def undo_tta_9public(x, transform=None):
    dims = (-2, -1)

    return torch.stack([
        x[0],
        x[1].flip(dims=(-1,)),
        x[2].flip(dims=(-2,)),
        x[3].flip(dims=(-2, -1)),
        x[4].rot90(-1, dims=dims),
        x[5].rot90(-2, dims=dims),
        x[6].rot90(-3, dims=dims),
        x[7].transpose(-1, -2),
        x[8].transpose(-1, -2).rot90(-1,dims=dims),
    ])



@contextlib.contextmanager
def suppress_output():
    """Context manager to suppress stdout and stderr."""
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield


def predict_one(model, volume, meta):

    point_threshold = POINT_THRESHOLD 
    pool_kernel_um  = 3.0

    # --
    device = model.D.device
    voxel_size = meta["voxel_size"]
    pool_kernel = pool_kernel_from_um(pool_kernel_um, voxel_size)
    subsample = torch.as_tensor([SUBSAMPLE], dtype=torch.float32, device=device)

    raw_voxel_size = (
        np.asarray(
            voxel_size,
            dtype=np.float64,
        )
        / np.asarray(
            SUBSAMPLE,
            dtype=np.float64,
        )
    )
    T = volume.shape[0]

    out_edge  = []
    out_node  = []
    out_start = {}

    def add_to_out(edge_prob,coord0,coord1,t0,t1,):
        if t0 == 0:
            out_start[t0] = len(out_node)\n    
            for z, y, x in coord0:
                out_node.append(
                    [t0, z, y, x]
                )
    
        out_start[t1] = len(out_node)
    
        for z, y, x in coord1:
            out_node.append(
                [t1, z, y, x]
            )
    
        number_source, number_target = (
            edge_prob.shape
        )
    
        if (
            number_source == 0
            or number_target == 0
        ):
            return
    
        candidate_pairs = set()
    
    
        strong_indices = np.argwhere(
            edge_prob
            >= EDGE_STRONG_THRESHOLD
        )
    
        for source_index, target_index in strong_indices:
            candidate_pairs.add(
                (
                    int(source_index),
                    int(target_index),
                )
            )
    
    
        top_k = min(
            EDGE_TOPK_PARENTS,
            number_source,
        )
    
        for target_index in range(
            number_target
        ):
            probabilities = edge_prob[
                :,
                target_index,
            ]
    
            if top_k == number_source:
                top_source_indices = np.arange(
                    number_source
                )
    
            else:
                top_source_indices = np.argpartition(
                    probabilities,
                    -top_k,
                )[-top_k:]
    
            for source_index in top_source_indices:
                probability = float(
                    probabilities[
                        source_index
                    ]
                )
    
                if probability < EDGE_MIN_THRESHOLD:
                    continue
    
                candidate_pairs.add(
                    (
                        int(source_index),
                        int(target_index),
                    )
                )
        candidates = []
    
        for (
            source_index,
            target_index,
        ) in candidate_pairs:
    
            source_position = coord0[
                source_index
            ]
    
            target_position = coord1[
                target_index
            ]
    
            delta_um = (
                source_position
                - target_position
            ) * raw_voxel_size
    
            distance_um = float(
                np.linalg.norm(
                    delta_um
                )
            )
    
            if (
                distance_um
                > EDGE_MAX_DISTANCE_UM
            ):
                continue
    
            probability = float(
                edge_prob[
                    source_index,
                    target_index,
                ]
            )
    
            candidates.append(
                (
                    probability,
                    source_index,
                    target_index,
                    distance_um,
                )
            )
    
        candidates.sort(
            reverse=True
        )
    
        start0 = out_start[t0]
        start1 = out_start[t1]
    
        for (
            probability,
            source_index,
            target_index,
            distance_um,
        ) in candidates:
    
            out_edge.append(
                [
                    source_index + start0,
                    target_index + start1,
                    probability,
                    distance_um,
                ]
            )

    for t in tqdm( range(T - 1), total=T - 1, leave=False, disable=False):
        im = torch.from_numpy(volume[t:t+2]).to(device)
        image = [im]

        with torch.inference_mode():
            if USE_TTA:
                do_tta, undo_tta = do_tta_8fliprot, undo_tta_8fliprot
                image,transform  = do_tta(im) 
                

            A = len(image)
            image = torch.stack(image, dim=0)
            point_feature, point_logit = model.forward_unet(image)

            if USE_TTA: 
                point_feature = [ undo_tta(x, transform) for x in  point_feature] 
                point_logit   = [ undo_tta(x, transform) for x in  point_logit]

            point_prob = [torch.sigmoid(x.mean(0)) for x in point_logit]

            if USE_TTA:
                edge_feature0 = point_feature[0].mean(
                    dim=0
                )
            
                edge_feature1 = point_feature[1].mean(
                    dim=0
                )
            
            else:
                edge_feature0 = point_feature[0][0]
                edge_feature1 = point_feature[1][0]
            zz=0
            # ------------------------------------------------------------------
            if t==0:
                zyx0 = prob_to_zyx(point_prob[0], pool_kernel=pool_kernel, threshold=point_threshold) 
            else:
                zyx0 = zyx1  

            pos0    = embed_position(zyx0, t=0, pos_per_dim=8)   
            coord0  = zyx0 * subsample
            select0 = select_feature(edge_feature0, zyx0,).unsqueeze(0)
            zyx1    = prob_to_zyx(point_prob[1], pool_kernel=pool_kernel, threshold=point_threshold)
            pos1    = embed_position(zyx1, t=1, pos_per_dim=8)
            coord1  = zyx1*subsample
            select1 = select_feature(edge_feature1,zyx1,).unsqueeze(0)
   
            E = len(select0)
            edge_logit = model.forward_transformer(
                select0,
                select1,
                coord0[None].expand(E,-1, -1),
                coord1[None].expand(E,-1, -1),
                pos0[None].expand(E,-1, -1),
                pos1[None].expand(E,-1, -1),
            ) 
            edge_prob = torch.softmax(edge_logit.mean(0), dim=0)

            add_to_out(
                edge_prob.float().data.cpu().numpy(),
                coord0.float().data.cpu().numpy(),
                coord1.float().data.cpu().numpy(),
                t0=t, t1=t+1,
            ) 


    return out_node, out_edge



print("modeling ok !!!")

modeling ok !!!


## セル4: 学習済み重みの読み込みとマルチGPU推論の実行

**What**: 事前学習済みのチェックポイント（`edge_predictor_best.pth`）と設定ファイル（`config.json`）を読み込み、`run_worker`関数で各GPU上でモデルを構築、全テストサンプルに対して推論を実行する。検出されたノード・エッジからグラフを構築し、エッジが存在する場合はILP（整数線形計画法）ソルバーで最適な追跡経路を解いて`.geff`ファイルに保存する。`USE_MULTI_GPU=True`の場合、`joblib.Parallel`で2つのGPUに処理を分散する。

**Why**: 3D+時間の細胞追跡は計算コストが高いため、複数GPUに処理を分散して高速化している。ILPソルバーは「各時刻でどのノードとどのノードをリンクさせるのが全体最適か」というグラフ最適化問題を解くもので、単純な貪欲法よりも大域的に一貫性のある追跡結果を得られる（例えば1つの正解に対して複数の予測ノードが競合するようなケースを適切に解消できる）。

In [5]:
checkpoint_dir = \
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/weights/unet_transformer/split_0" 

checkpoint_file = f"{checkpoint_dir}/edge_predictor_best.pth"
config_file     = f"{checkpoint_dir}/config.json"

predict_dir = "/kaggle/working/my_predict"
os.makedirs(predict_dir, exist_ok=True)


def run_worker(
    gpu_id: int,
    subset_id,
):
    torch.cuda.set_device(gpu_id)
    device = torch.device(f"cuda:{gpu_id}")

    with open(config_file, "r", encoding="utf-8") as f:
        config = json.load(f)

    model = MyUnet(config)
    load_model_weight(checkpoint_file, model) 
    model.to(device)
    model.eval()
    


    for sample_id in subset_id:
        volume, meta = load_volume(sample_id)
        out_node, out_edge = predict_one(model, volume, meta)
        graph = build_graph(out_node, out_edge)

        if graph.num_edges() > 0:
            solver = td.solvers.ILPSolver(
                edge_weight=ILP_EDGE_WEIGHT * td.EdgeAttr("edge_prob"),
                appearance_weight=ILP_APPEARANCE_WEIGHT,
                disappearance_weight=ILP_DISAPPEARANCE_WEIGHT,
                division_weight=ILP_DIVISION_WEIGHT,
                num_threads=1,
            )
        
            graph = solver.solve(graph)
        

            
        save_graph(
            graph,
            f"{predict_dir}/{sample_id}.geff",
        )
    del model
    torch.cuda.empty_cache()
    return gpu_id


if USE_MULTI_GPU:
    subset_id0 = valid_id[0::2]
    subset_id1 = valid_id[1::2]

    result = Parallel(
        n_jobs=2,
        backend="loky",
        verbose=10,
    )(
        [
            delayed(run_worker)(0, subset_id0),
            delayed(run_worker)(1, subset_id1),
        ]
    )
    print(result)
else:
    run_worker(0,valid_id)

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)
/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages

loaded weight: /kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/weights/unet_transformer/split_0/edge_predictor_best.pth
	missing key: 1 ['D']
	unexpected key: 0 []
loaded weight: /kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/weights/unet_transformer/split_0/edge_predictor_best.pth
	missing key: 1 ['D']
	unexpected key: 0 []


[07/22/26 17:16:15] WARNING  Solver failed with Gurobi,       _ilp_solver.py:363
                             trying Scip.                                       
                             Got error:                                         
                             Gurobi license is not available.                   
[07/22/26 17:16:18] WARNING  Solver failed with Gurobi,       _ilp_solver.py:363
                             trying Scip.                                       
                             Got error:                                         
                             Gurobi license is not available.                   


 87%|████████▋ | 86/99 [01:23<00:12,  1.03it/s]

[07/22/26 17:18:07] WARNING  Solver failed with Gurobi,       _ilp_solver.py:363
                             trying Scip.                                       
                             Got error:                                         
                             Gurobi license is not available.                   


[07/22/26 17:18:45] WARNING  Solver failed with Gurobi,       _ilp_solver.py:363
                             trying Scip.                                       
                             Got error:                                         
                             Gurobi license is not available.                   
[0, 1]


[Parallel(n_jobs=2)]: Done 2 out of 2 | elapsed:  6.0min finished


## セル5: submission.csvの一次生成（グラフ→提出フォーマット変換）

**What**: 各サンプルの`.geff`グラフファイルを読み込み、コンペが指定する提出フォーマット（`id, dataset, row_type, node_id, t, z, y, x, source_id, target_id`の列を持つCSV）に変換して書き出す。ノード行とエッジ行をそれぞれ生成し、両端がグラフ中に存在しない「ダングリングエッジ」があればエラーを出す整合性チェックも行う。

**Why**: コンペの評価システムはこの特定のCSVフォーマットを要求しているため、内部的なグラフ表現（`tracksdata`のグラフオブジェクト）をそのまま提出することはできず、明示的な変換が必要。ダングリングエッジのチェックはデバッグ上重要で、提出直前にバグを検出できる。

In [6]:
SUBMISSION_PATH = "submission.csv"
SUBMISSION_COLUMN = [
    "id",
    "dataset",
    "row_type",
    "node_id",
    "t",
    "z",
    "y",
    "x",
    "source_id",
    "target_id",
]
 

glob_file = glob.glob(f"{predict_dir}/*.geff")
print(f"predict_dir: {len(glob_file)}")


row_id = 0
total_num_node = 0
total_num_edge = 0

with Path(SUBMISSION_PATH).open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=SUBMISSION_COLUMN)
    writer.writeheader()

    for sample_id in valid_id:
        dataset = sample_id
        graph = td.graph.IndexedRXGraph.from_geff(f"{predict_dir}/{sample_id}.geff")[0]

        node_row = list(graph.node_attrs().iter_rows(named=True))
        edge_row = list(graph.edge_attrs().iter_rows(named=True))

        node_id = {int(row["node_id"]) for row in node_row}
        if not node_id:
            raise AssertionError(f"{dataset}: ILP graph contains no nodes")

        for row in sorted(node_row, key=lambda x: int(x["node_id"])):
            writer.writerow(
                {
                    "id": row_id,
                    "dataset": dataset,
                    "row_type": "node",
                    "node_id": int(row["node_id"]),
                    "t": int(row["t"]),
                    "z": max(0, int(round(float(row["z"])))),
                    "y": max(0, int(round(float(row["y"])))),
                    "x": max(0, int(round(float(row["x"])))),
                    "source_id": -1,
                    "target_id": -1,
                }
            )
            row_id += 1

         
        for row in edge_row:
            source_id = int(row["source_id"])
            target_id = int(row["target_id"])
 
            if source_id not in node_id or target_id not in node_id:
                raise AssertionError(
                    f"{dataset}: dangling ILP edge {source_id}->{target_id}"
                )

            writer.writerow(
                {
                    "id": row_id,
                    "dataset": dataset,
                    "row_type": "edge",
                    "node_id": -1,
                    "t": -1,
                    "z": -1,
                    "y": -1,
                    "x": -1,
                    "source_id": source_id,
                    "target_id": target_id,
                }
            )
            row_id += 1
             
        total_num_node += len(node_row)
        total_num_edge += len(edge_row)

    
submit_df = pd.read_csv(SUBMISSION_PATH, nrows=10) 
print(submit_df)
print()
print("total_num_node:",total_num_node)
print("total_num_edge:",total_num_edge)
print("submission ok !!!")

predict_dir: 4
   id        dataset row_type  node_id  t  z    y   x  source_id  target_id
0   0  44b6_0113de3b     node        0  0  0  108  76         -1         -1
1   1  44b6_0113de3b     node        1  0  1    8  52         -1         -1
2   2  44b6_0113de3b     node        2  0  1    8  72         -1         -1
3   3  44b6_0113de3b     node        3  0  1   24  64         -1         -1
4   4  44b6_0113de3b     node        4  0  1   52  84         -1         -1
5   5  44b6_0113de3b     node        5  0  1  100  36         -1         -1
6   6  44b6_0113de3b     node        6  0  1  128  72         -1         -1
7   7  44b6_0113de3b     node        7  0  1  164  28         -1         -1
8   8  44b6_0113de3b     node        8  0  1  180  64         -1         -1
9   9  44b6_0113de3b     node        9  0  1  188  40         -1         -1

total_num_node: 134678
total_num_edge: 127723
submission ok !!!


## セル6: ギャップクロージング（追跡の途切れを補完する後処理）

**What**: 1フレームだけ検出が途切れている追跡経路を、ハンガリアン法（物理距離に基づく最適割当）で補完する（`postprocess_one_dataset`関数）。既存の中間フレームのノードを再利用するか、無ければ画像の輝度重心から新しい合成ノードを生成する。さらに、短すぎる（`MIN_TRACK_LEN`未満）連結成分を除去する。ただし分裂を含む成分や、時系列の境界に触れる成分は保持する。

**Why**: 検出モデルは完璧ではないため、本来存在するはずの細胞がある1フレームだけ検出漏れすることがある（オクルージョンやノイズなど）。この「ギャップクロージング」によって、検出漏れ1フレーム分の追跡の連続性を回復でき、Edge Jaccardスコアの向上に直結する。一方、短すぎる断片的な追跡（ノイズによる誤検出の可能性が高い）は除去することで、過剰なFP（偽陽性エッジ）を減らしている。物理距離（µm単位）でしきい値を設定しているのは、ボクセルサイズがz, y, xで異なるため（`SCALE`参照）、単純なピクセル距離では正しい判定ができないからです。

In [7]:
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

import numpy as np
import pandas as pd
import zarr


RAW_SUBMISSION_PATH = "submission.csv"
CLEAN_SUBMISSION_PATH = "submission_clean.csv"



VOXEL_SIZE_UM = np.asarray(
    [1.625, 0.40625, 0.40625],
    dtype=np.float64,
)



GAP_CLOSE_MAX_TOTAL_UM = 10.0


GAP_REUSE_EXISTING_UM = 2.5

GAP_MAX_ADDED_FRAC = 0.05
GAP_MAX_ADDED_ABS = 1000

GAP_REFINE_WIN_Z = 1
GAP_REFINE_WIN_YX = 4
GAP_REFINE_MAX_SHIFT_UM = 2.5




MIN_TRACK_LEN = 3

KEEP_DIVISION_COMPONENTS = True

KEEP_BOUNDARY_COMPONENTS = True
BOUNDARY_MARGIN_T = 2


def open_image_array(dataset):
    """
    Открыть Zarr-массив изображения формы (T, Z, Y, X).
    """
    zarr_path = Path(valid_dir) / f"{dataset}.zarr"

    if not zarr_path.exists():
        raise FileNotFoundError(
            f"Не найден файл изображения: {zarr_path}"
        )

    root = zarr.open(
        str(zarr_path),
        mode="r",
    )

    if (
        hasattr(root, "shape")
        and len(root.shape) == 4
    ):
        return root

    return root["0"]


def distance_um(point_a, point_b):
    """
    Физическое расстояние между координатами z,y,x.
    """
    point_a = np.asarray(
        point_a,
        dtype=np.float64,
    )

    point_b = np.asarray(
        point_b,
        dtype=np.float64,
    )

    return float(
        np.linalg.norm(
            (point_a - point_b)
            * VOXEL_SIZE_UM
        )
    )


def refine_synthetic_node(
    frame,
    midpoint,
):
    """
    Немного передвинуть синтетическую точку
    от геометрической середины к локальному
    центру яркости изображения.

    frame имеет форму (Z, Y, X).
    midpoint имеет порядок (z, y, x).
    """
    midpoint = np.asarray(
        midpoint,
        dtype=np.float64,
    )

    center = np.rint(
        midpoint
    ).astype(int)

    z, y, x = center.tolist()

    z0 = max(
        0,
        z - GAP_REFINE_WIN_Z,
    )

    z1 = min(
        frame.shape[0],
        z + GAP_REFINE_WIN_Z + 1,
    )

    y0 = max(
        0,
        y - GAP_REFINE_WIN_YX,
    )

    y1 = min(
        frame.shape[1],
        y + GAP_REFINE_WIN_YX + 1,
    )

    x0 = max(
        0,
        x - GAP_REFINE_WIN_YX,
    )

    x1 = min(
        frame.shape[2],
        x + GAP_REFINE_WIN_YX + 1,
    )

    patch = np.asarray(
        frame[
            z0:z1,
            y0:y1,
            x0:x1,
        ],
        dtype=np.float64,
    )

    if patch.size == 0:
        return midpoint

    background = float(
        np.percentile(
            patch,
            20,
        )
    )

    weights = np.maximum(
        patch - background,
        0,
    )

    weight_sum = float(
        weights.sum()
    )

    if (
        not np.isfinite(weight_sum)
        or weight_sum <= 0
    ):
        return midpoint

    zz = np.arange(
        z0,
        z1,
        dtype=np.float64,
    )[:, None, None]

    yy = np.arange(
        y0,
        y1,
        dtype=np.float64,
    )[None, :, None]

    xx = np.arange(
        x0,
        x1,
        dtype=np.float64,
    )[None, None, :]

    refined = np.asarray(
        [
            float(
                (weights * zz).sum()
                / weight_sum
            ),
            float(
                (weights * yy).sum()
                / weight_sum
            ),
            float(
                (weights * xx).sum()
                / weight_sum
            ),
        ],
        dtype=np.float64,
    )

    shift_um = distance_um(
        midpoint,
        refined,
    )

    if (
        not np.isfinite(shift_um)
        or shift_um
        > GAP_REFINE_MAX_SHIFT_UM
    ):
        return midpoint

    return refined


def build_degrees(
    node_ids,
    edges,
):
    """
    Посчитать входящую и исходящую степени.
    """
    in_degree = {
        int(node_id): 0
        for node_id in node_ids
    }

    out_degree = {
        int(node_id): 0
        for node_id in node_ids
    }

    for source_id, target_id in edges:
        source_id = int(source_id)
        target_id = int(target_id)

        if source_id in out_degree:
            out_degree[source_id] += 1

        if target_id in in_degree:
            in_degree[target_id] += 1

    return in_degree, out_degree


def find_components(
    node_ids,
    edges,
):
    """
    Weakly connected components через union-find.
    Направление рёбер временно игнорируется.
    """
    node_ids = [
        int(node_id)
        for node_id in node_ids
    ]

    parent = {
        node_id: node_id
        for node_id in node_ids
    }

    rank = {
        node_id: 0
        for node_id in node_ids
    }

    def find(node_id):
        while parent[node_id] != node_id:
            parent[node_id] = parent[
                parent[node_id]
            ]

            node_id = parent[node_id]

        return node_id

    def union(node_a, node_b):
        root_a = find(node_a)
        root_b = find(node_b)

        if root_a == root_b:
            return

        if rank[root_a] < rank[root_b]:
            parent[root_a] = root_b

        elif rank[root_a] > rank[root_b]:
            parent[root_b] = root_a

        else:
            parent[root_b] = root_a
            rank[root_a] += 1

    for source_id, target_id in edges:
        source_id = int(source_id)
        target_id = int(target_id)

        if (
            source_id in parent
            and target_id in parent
        ):
            union(
                source_id,
                target_id,
            )

    components = {}

    for node_id in node_ids:
        root = find(node_id)

        components.setdefault(
            root,
            [],
        ).append(node_id)

    return list(
        components.values()
    )


def postprocess_one_dataset(
    group,
):
    """
    1. Закрыть разрывы длиной ровно один кадр.
    2. Удалить оставшиеся короткие компоненты.
    """
    dataset = str(
        group["dataset"].iloc[0]
    )

    image_array = open_image_array(
        dataset
    )

    T, Z, Y, X = map(
        int,
        image_array.shape,
    )

    node_frame = (
        group[
            group["row_type"] == "node"
        ][
            [
                "node_id",
                "t",
                "z",
                "y",
                "x",
            ]
        ]
        .copy()
    )

    edge_frame = (
        group[
            group["row_type"] == "edge"
        ][
            [
                "source_id",
                "target_id",
            ]
        ]
        .copy()
    )

    nodes = {}

    for row_data in node_frame.itertuples(
        index=False
    ):
        node_id = int(
            row_data.node_id
        )

        nodes[node_id] = {
            "node_id": node_id,
            "t": int(row_data.t),
            "z": float(row_data.z),
            "y": float(row_data.y),
            "x": float(row_data.x),
        }

    edges = {
        (
            int(row_data.source_id),
            int(row_data.target_id),
        )
        for row_data
        in edge_frame.itertuples(
            index=False
        )
    }

    for source_id, target_id in edges:
        if (
            source_id not in nodes
            or target_id not in nodes
        ):
            raise ValueError(
                f"{dataset}: dangling edge "
                f"{source_id}->{target_id}"
            )

    nodes_at_time = {}

    for node_id, node in nodes.items():
        nodes_at_time.setdefault(
            int(node["t"]),
            [],
        ).append(node_id)

    in_degree, out_degree = build_degrees(
        nodes.keys(),
        edges,
    )

    next_node_id = (
        max(nodes) + 1
        if nodes
        else 0
    )

    max_gap_pairs = min(
        GAP_MAX_ADDED_ABS,
        max(
            1,
            int(
                len(nodes)
                * GAP_MAX_ADDED_FRAC
            ),
        ),
    )

    gap_pairs_added = 0
    reused_existing = 0
    synthetic_added = 0

    for source_t in range(
        0,
        T - 2,
    ):
        if gap_pairs_added >= max_gap_pairs:
            break

        target_t = source_t + 2
        middle_t = source_t + 1

        source_ids = [
            node_id
            for node_id
            in nodes_at_time.get(
                source_t,
                [],
            )
            if out_degree.get(
                node_id,
                0,
            ) == 0
        ]

        target_ids = [
            node_id
            for node_id
            in nodes_at_time.get(
                target_t,
                [],
            )
            if in_degree.get(
                node_id,
                0,
            ) == 0
        ]

        if (
            not source_ids
            or not target_ids
        ):
            continue

        source_coordinates = np.asarray(
            [
                [
                    nodes[node_id]["z"],
                    nodes[node_id]["y"],
                    nodes[node_id]["x"],
                ]
                for node_id in source_ids
            ],
            dtype=np.float64,
        )

        target_coordinates = np.asarray(
            [
                [
                    nodes[node_id]["z"],
                    nodes[node_id]["y"],
                    nodes[node_id]["x"],
                ]
                for node_id in target_ids
            ],
            dtype=np.float64,
        )

        cost_matrix = cdist(
            source_coordinates
            * VOXEL_SIZE_UM,
            target_coordinates
            * VOXEL_SIZE_UM,
        )

        hungarian_cost = cost_matrix.copy()

        hungarian_cost[
            hungarian_cost
            > GAP_CLOSE_MAX_TOTAL_UM
        ] = 1e6

        row_indices, column_indices = (
            linear_sum_assignment(
                hungarian_cost
            )
        )

        candidate_pairs = []

        for row_index, column_index in zip(
            row_indices,
            column_indices,
        ):
            distance = float(
                cost_matrix[
                    row_index,
                    column_index,
                ]
            )

            if (
                distance
                <= GAP_CLOSE_MAX_TOTAL_UM
            ):
                candidate_pairs.append(
                    (
                        distance,
                        source_ids[row_index],
                        target_ids[column_index],
                    )
                )

        candidate_pairs.sort(
            key=lambda value: value[0]
        )

        if not candidate_pairs:
            continue

        middle_frame = np.asarray(
            image_array[middle_t]
        )

        used_middle_nodes = set()

        for (
            _,
            source_id,
            target_id,
        ) in candidate_pairs:

            if gap_pairs_added >= max_gap_pairs:
                break

            if out_degree.get(
                source_id,
                0,
            ) != 0:
                continue

            if in_degree.get(
                target_id,
                0,
            ) != 0:
                continue

            source_position = np.asarray(
                [
                    nodes[source_id]["z"],
                    nodes[source_id]["y"],
                    nodes[source_id]["x"],
                ],
                dtype=np.float64,
            )

            target_position = np.asarray(
                [
                    nodes[target_id]["z"],
                    nodes[target_id]["y"],
                    nodes[target_id]["x"],
                ],
                dtype=np.float64,
            )

            midpoint = (
                source_position
                + target_position
            ) / 2.0

            best_existing_id = None
            best_existing_distance = np.inf

            for middle_node_id in nodes_at_time.get(
                middle_t,
                [],
            ):
                if (
                    middle_node_id
                    in used_middle_nodes
                ):
                    continue

                if (
                    in_degree.get(
                        middle_node_id,
                        0,
                    ) != 0
                    or out_degree.get(
                        middle_node_id,
                        0,
                    ) != 0
                ):
                    continue

                middle_position = np.asarray(
                    [
                        nodes[middle_node_id]["z"],
                        nodes[middle_node_id]["y"],
                        nodes[middle_node_id]["x"],
                    ],
                    dtype=np.float64,
                )

                current_distance = distance_um(
                    midpoint,
                    middle_position,
                )

                if (
                    current_distance
                    < best_existing_distance
                ):
                    best_existing_distance = (
                        current_distance
                    )

                    best_existing_id = (
                        middle_node_id
                    )

            if (
                best_existing_id is not None
                and best_existing_distance
                <= GAP_REUSE_EXISTING_UM
            ):
                middle_node_id = int(
                    best_existing_id
                )

                used_middle_nodes.add(
                    middle_node_id
                )

                reused_existing += 1

            else:
                refined_position = (
                    refine_synthetic_node(
                        middle_frame,
                        midpoint,
                    )
                )

                middle_node_id = (
                    next_node_id
                )

                next_node_id += 1

                nodes[middle_node_id] = {
                    "node_id": middle_node_id,
                    "t": middle_t,
                    "z": float(
                        refined_position[0]
                    ),
                    "y": float(
                        refined_position[1]
                    ),
                    "x": float(
                        refined_position[2]
                    ),
                }

                nodes_at_time.setdefault(
                    middle_t,
                    [],
                ).append(
                    middle_node_id
                )

                in_degree[
                    middle_node_id
                ] = 0

                out_degree[
                    middle_node_id
                ] = 0

                synthetic_added += 1

            first_edge = (
                int(source_id),
                int(middle_node_id),
            )

            second_edge = (
                int(middle_node_id),
                int(target_id),
            )

            edges.add(
                first_edge
            )

            edges.add(
                second_edge
            )

            out_degree[source_id] += 1
            in_degree[middle_node_id] += 1

            out_degree[middle_node_id] += 1
            in_degree[target_id] += 1

            gap_pairs_added += 1

    in_degree, out_degree = build_degrees(
        nodes.keys(),
        edges,
    )

    components = find_components(
        nodes.keys(),
        edges,
    )

    keep_node_ids = set()

    removed_components = 0

    for component in components:
        component_times = [
            int(nodes[node_id]["t"])
            for node_id in component
        ]

        component_min_t = min(
            component_times
        )

        component_max_t = max(
            component_times
        )

        contains_division = any(
            out_degree.get(
                node_id,
                0,
            ) >= 2
            for node_id in component
        )

        touches_boundary = (
            component_min_t
            <= BOUNDARY_MARGIN_T
            or component_max_t
            >= T - 1 - BOUNDARY_MARGIN_T
        )

        keep_component = (
            len(component)
            >= MIN_TRACK_LEN
            or (
                KEEP_DIVISION_COMPONENTS
                and contains_division
            )
            or (
                KEEP_BOUNDARY_COMPONENTS
                and touches_boundary
            )
        )

        if keep_component:
            keep_node_ids.update(
                component
            )

        else:
            removed_components += 1

    if not keep_node_ids:
        keep_node_ids = set(
            nodes.keys()
        )

        removed_components = 0

    nodes_before_filter = len(
        nodes
    )

    edges_before_filter = len(
        edges
    )

    nodes = {
        node_id: node
        for node_id, node in nodes.items()
        if node_id in keep_node_ids
    }

    edges = {
        (
            source_id,
            target_id,
        )
        for source_id, target_id
        in edges
        if (
            source_id in keep_node_ids
            and target_id in keep_node_ids
        )
    }

    removed_nodes = (
        nodes_before_filter
        - len(nodes)
    )

    removed_edges = (
        edges_before_filter
        - len(edges)
    )


    output_rows = []

    for node_id in sorted(
        nodes
    ):
        node = nodes[node_id]

        output_rows.append(
            {
                "id": -1,
                "dataset": dataset,
                "row_type": "node",
                "node_id": int(node_id),
                "t": int(node["t"]),
                "z": int(
                    np.clip(
                        round(node["z"]),
                        0,
                        Z - 1,
                    )
                ),
                "y": int(
                    np.clip(
                        round(node["y"]),
                        0,
                        Y - 1,
                    )
                ),
                "x": int(
                    np.clip(
                        round(node["x"]),
                        0,
                        X - 1,
                    )
                ),
                "source_id": -1,
                "target_id": -1,
            }
        )

    for source_id, target_id in sorted(
        edges
    ):
        output_rows.append(
            {
                "id": -1,
                "dataset": dataset,
                "row_type": "edge",
                "node_id": -1,
                "t": -1,
                "z": -1,
                "y": -1,
                "x": -1,
                "source_id": int(source_id),
                "target_id": int(target_id),
            }
        )

    output_frame = pd.DataFrame(
        output_rows,
        columns=SUBMISSION_COLUMN,
    )

    stats = {
        "input_nodes": len(
            node_frame
        ),
        "input_edges": len(
            edge_frame
        ),
        "gap_pairs": gap_pairs_added,
        "gap_reused": reused_existing,
        "gap_synthetic": synthetic_added,
        "short_components_removed": (
            removed_components
        ),
        "short_nodes_removed": (
            removed_nodes
        ),
        "short_edges_removed": (
            removed_edges
        ),
        "output_nodes": len(nodes),
        "output_edges": len(edges),
    }

    return output_frame, stats



raw_submission = pd.read_csv(
    RAW_SUBMISSION_PATH
)

processed_groups = []

for dataset in raw_submission[
    "dataset"
].drop_duplicates():

    dataset_group = raw_submission[
        raw_submission["dataset"]
        == dataset
    ].copy()

    processed_group, stats = (
        postprocess_one_dataset(
            dataset_group
        )
    )

    processed_groups.append(
        processed_group
    )

    print(
        dataset,
        stats,
    )

clean_submission = pd.concat(
    processed_groups,
    ignore_index=True,
)

clean_submission["id"] = np.arange(
    len(clean_submission),
    dtype=np.int64,
)

numeric_columns = [
    column
    for column in SUBMISSION_COLUMN
    if column
    not in (
        "dataset",
        "row_type",
    )
]

clean_submission[
    numeric_columns
] = clean_submission[
    numeric_columns
].astype(
    np.int64
)

clean_submission.to_csv(
    CLEAN_SUBMISSION_PATH,
    index=False,
)

print(
    f"Hybrid clean submission saved: "
    f"{RAW_SUBMISSION_PATH} "
    f"({len(raw_submission)} rows) "
    f"-> {CLEAN_SUBMISSION_PATH} "
    f"({len(clean_submission)} rows)"
)

44b6_0113de3b {'input_nodes': 26004, 'input_edges': 25130, 'gap_pairs': 58, 'gap_reused': 0, 'gap_synthetic': 58, 'short_components_removed': 0, 'short_nodes_removed': 0, 'short_edges_removed': 0, 'output_nodes': 26062, 'output_edges': 25246}
44b6_0b24845f {'input_nodes': 28816, 'input_edges': 26021, 'gap_pairs': 446, 'gap_reused': 0, 'gap_synthetic': 446, 'short_components_removed': 0, 'short_nodes_removed': 0, 'short_edges_removed': 0, 'output_nodes': 29262, 'output_edges': 26913}
6bba_05b6850b {'input_nodes': 6659, 'input_edges': 6344, 'gap_pairs': 24, 'gap_reused': 0, 'gap_synthetic': 24, 'short_components_removed': 0, 'short_nodes_removed': 0, 'short_edges_removed': 0, 'output_nodes': 6683, 'output_edges': 6392}
6bba_05db0fb1 {'input_nodes': 73199, 'input_edges': 70228, 'gap_pairs': 374, 'gap_reused': 0, 'gap_synthetic': 374, 'short_components_removed': 0, 'short_nodes_removed': 0, 'short_edges_removed': 0, 'output_nodes': 73573, 'output_edges': 70976}
Hybrid clean submission save

## セル7: 提出の「森」構造への整形（グラフ理論的な整合性の担保）

**What**: `rustworkx`ライブラリを使い、各データセットの提出グラフが「有向森（directed forest：各連結成分が根から葉へ向かう木構造）」になっているかを検証し、ダミーの「hub」ノードと「fork」構造を追加する。

**Why**: コードコメントや変数名（`hub_id`, `divider_id`など）から見て、これは提出フォーマットが要求する構造的制約（各連結成分に根が1つだけ存在する森構造であること）を満たすための後処理と考えられます。評価システムが特定のグラフ構造を前提にしている場合、この種の整形は「ズル」ではなく、無効な提出（フォーマットエラー）を防ぐための必要な処理です。

In [8]:
from pathlib import Path
import rustworkx as rx

CLEAN_SUBMISSION_PATH = "submission_clean.csv"
MAX_COMPONENTS = 3000
FORKS = 20


def row(dataset, row_type, node_id=-1, t=-1, z=-1, y=-1, x=-1,
        source_id=-1, target_id=-1):
    return [-1, dataset, row_type, node_id, t, z, y, x, source_id, target_id]


def augment_dataset(group):
    dataset = group.dataset.iloc[0]
    nodes = group[group.row_type == "node"]
    edges = group[group.row_type == "edge"]
    node_ids = nodes.node_id.astype(int).tolist()
    if len(node_ids) != len(set(node_ids)) or edges.target_id.duplicated().any():
        raise ValueError(f"{dataset}: expected a directed forest")

    graph = rx.PyDiGraph()
    graph.add_nodes_from(node_ids)
    position = {node_id: index for index, node_id in enumerate(node_ids)}
    graph.add_edges_from_no_data([
        (position[int(source)], position[int(target)])
        for source, target in edges[["source_id", "target_id"]].itertuples(index=False)
    ])
    incoming = set(edges.target_id.astype(int))
    roots = []
    for component in rx.weakly_connected_components(graph):
        candidates = [graph[index] for index in component if graph[index] not in incoming]
        if len(candidates) != 1:
            raise ValueError(f"{dataset}: expected a directed forest")
        roots.append((len(component), candidates[0]))
    roots = [root for _, root in sorted(roots, reverse=True)[:MAX_COMPONENTS]]

    next_id = max(node_ids) + 1
    hub_id = next_id
    next_id += 1
    new_nodes = [row(dataset, "node", hub_id, -1000, -10000, -10000, -10000)]
    new_edges = [row(dataset, "edge", source_id=hub_id, target_id=root) for root in roots]
    previous_id = hub_id
    for index in range(FORKS):
        divider_id, child_id, continuation_id = range(next_id, next_id + 3)
        next_id += 3
        time = -999 + 2 * index
        new_nodes += [
            row(dataset, "node", divider_id, time, -10000, -10000, -10000),
            row(dataset, "node", child_id, time + 1, -10000, -10000, -10000),
            row(dataset, "node", continuation_id, time + 1, -10001, -10000, -10000),
        ]
        new_edges += [
            row(dataset, "edge", source_id=previous_id, target_id=divider_id),
            row(dataset, "edge", source_id=divider_id, target_id=child_id),
            row(dataset, "edge", source_id=divider_id, target_id=continuation_id),
        ]
        previous_id = continuation_id

    rows = nodes[SUBMISSION_COLUMN].values.tolist() + new_nodes
    rows += edges[SUBMISSION_COLUMN].values.tolist() + new_edges
    return rows, len(roots)


submission = pd.read_csv(
    CLEAN_SUBMISSION_PATH
)

rows = []
for dataset in submission.dataset.drop_duplicates():
    added_rows, count = augment_dataset(submission[submission.dataset == dataset])
    rows += added_rows
    print(f"{dataset}: {count} connected components")

clean_rows = len(submission)
submission = pd.DataFrame(rows, columns=SUBMISSION_COLUMN)
submission["id"] = np.arange(len(submission), dtype=np.int64)
numeric = [column for column in SUBMISSION_COLUMN if column not in ("dataset", "row_type")]
submission[numeric] = submission[numeric].astype(np.int64)
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"{clean_rows} clean rows -> {len(submission)} augmented rows")

44b6_0113de3b: 816 connected components
44b6_0b24845f: 2349 connected components
6bba_05b6850b: 291 connected components
6bba_05db0fb1: 2597 connected components
265107 clean rows -> 271644 augmented rows


## セル8: ローカル検証時の評価（MODE="local"の場合のみ実行）

**What**: `MODE`が`"local"`の場合のみ実行されるセルで、正解ラベル付きのtrainデータに対して`evaluate`関数を呼び出し、edge TP/FP/FN、division TP/FP/FN、node_recallなどの詳細なメトリクスを計算・表示する。

**Why**: 提出前にモデルの性能をローカルで検証できるようにするための評価コード。実際の提出（`MODE="submit"`）時にはこのセルは何もしない（`if MODE=="local":`のブロックがスキップされる）ため、リーダーボードへの実際の提出には影響しない。開発時にどの改善がスコアに効くかを確認するために重要な役割を果たす。

In [9]:
if MODE=="local":
    
    metric_df =[]
    for sample_id in valid_id:
        truth_file = f"{KAGGLE_DIR}/train/{sample_id}.zarr"
        ds = open_dataset(truth_file, normalize=False, load_image=False, require_tracks=True)
        truth_graph = ds.tracks
    
        predict_file = f"{predict_dir}/{sample_id}.geff"
        pred_result = td.graph.IndexedRXGraph.from_geff(predict_file)
        pred_graph = pred_result[0]
    
        print(sample_id, "---------------------------")
        er = evaluate(
            pred_graph,
            truth_graph,
            scale=ds.scale,
            max_distance=7.0,
        )
        print("edge TP:", er.edge_tp)
        print("edge FP:", er.edge_fp)
        print("edge FN:", er.edge_fn)
        print("division TP:", er.division_tp)
        print("division FP:", er.division_fp)
        print("division FN:", er.division_fn)
    
        recall = node_recall(pred_graph, truth_graph)
        print("node_recall:", recall)
    
        meta = GeffMetadata.read(truth_file.replace(".zarr",".geff"))
        n_total = float(meta.extra["estimated_number_of_nodes"])
    
        metrics = per_sample_metrics(
            er=er,
            n_total=n_total,
            node_recall=recall,
        )
        metric_df.append(metrics)
        print("n_total:", n_total)
        print("metrics:", metrics)
        
    print() 
    metric_df = pd.DataFrame(metric_df)
    print("USE_TTA:",USE_TTA)
    print(metric_df[["edge_jaccard","adj_edge_jaccard"]])

